In [1]:
import glob
import os
from pathlib import Path

import dask.array as da
import tifffile as tff
import tqdm
import yaml
from multiview_stitcher import fusion
from multiview_stitcher import spatial_image_utils as si_utils
import boto3
import tqdm
import pandas as pd
import napari

#from napari_autoscape import convert_single_tile_to_ome_zarr, convert_tiles_to_stitched_ome_zarr
from ome_zarr import OMEZarrImage, OMEZarrMultiscale, OMEZarrScene
from ome_zarr_models.v06.coordinate_transforms import (
    Axis,
    CoordinateSystem,
)

from napari_autoscape import (
    sliding_window_inference,
    remove_duplicate_detections,
    extract_features
)

In [2]:
# viewer = napari.Viewer()

# client = Client()  # Dashboard at http://localhost:8787
# print(f"Dashboard: {client.dashboard_link}")

In [3]:
places = [
    #r'Z:\Collaborations\aif-botton-collaboration\Johannes\20260826\20260811_BF_W2_full_ImageStackFile_1',
    "/vol/data/SCAPE/20260826/20260811_BF_W2_full_ImageStackFile_1",
]

## Read tile images

In [6]:
imgs = []
transforms = []
for place in places:
    tif_files = glob.glob(os.path.join(place, "*.ome.tif"), recursive=True)

    sims = []
    translations = []
    metadata_file = tif_files[0].replace(".ome.tif", "_metadata.txt")
    with open(metadata_file) as f:
        metadata = yaml.safe_load(f)["Summary"]

    for tf in tqdm.tqdm(tif_files):

        tile_index = Path(tf).name.split("MMStack_")[1].replace(".ome.tif", "")

        # index positionlist by label
        positionlist = metadata["InitialPositionList"]
        positionlist = {position["Label"]: position for position in positionlist}
        position = positionlist[tile_index]

        scale = {
            "z": metadata["z-step_um"] if metadata["z-step_um"] != 0 else 1.0,
            "y": metadata["PixelSize_um"],
            "x": metadata["PixelSize_um"]
        }

        array = da.from_array(tff.imread(tf, is_ome=False, is_mmstack=False))
        size_y = array.shape[1] * scale["y"]
        size_x = array.shape[2] * scale["x"]
        size_z = array.shape[0] * scale["z"]

        translation = {
            "z": float(position["DeviceCoordinatesUm"]["ZStage:Z:32"][0]) - size_z // 2,
            "y": - float(position["DeviceCoordinatesUm"]["XYStage:XY:31"][1]) + size_y // 2,
            "x": float(position["DeviceCoordinatesUm"]["XYStage:XY:31"][0]) - size_x // 2,
        }
        translations.append(translation)

        sims.append(
            si_utils.get_sim_from_array(
                array,
                dims=["z", "y", "x"],
                scale=scale,
                translation=translation,
                transform_key="stage_metadata",
                )
        )

    image = fusion.fuse(sims, transform_key="stage_metadata", fusion_func=fusion.max_fusion)

    ngff_img = OMEZarrImage(image.data.squeeze(), axes="zyx", scale=scale,
                            axes_units={"z": "micrometer", "y": "micrometer", "x": "micrometer"}, name=Path(place).stem)
    ms_img = OMEZarrMultiscale(ngff_img, channel_names=["Brightfield"], channel_colors=["808080"])
    imgs.append(ms_img)

    z_min = min([t["z"] for t in translations])
    y_min = max([t["y"] for t in translations])
    x_min = min([t["x"] for t in translations])

    transforms.append({
        "type": "translation",
        "translation": (z_min, y_min, x_min),
        "input": {"name": "physical", "path": ms_img.name},
        "output": {"name": "world"}
    })

100%|██████████| 361/361 [00:07<00:00, 46.87it/s]


In [16]:
transforms[0]["translation"] = (-53.199999999999996, 2912.0161200000002 - size_y, 2902.18062)
transforms

[{'type': 'translation',
  'translation': (-53.199999999999996, 2503.69052, 2902.18062),
  'input': {'name': 'physical',
   'path': '20260811_BF_W2_full_ImageStackFile_1'},
  'output': {'name': 'world'}}]

In [ ]:
world_cs = CoordinateSystem(name="world", axes=(
    Axis(name="z", type="space", unit="micrometer"),
    Axis(name="y", type="space", unit="micrometer"),
    Axis(name="x", type="space", unit="micrometer"))
    )

scene = OMEZarrScene(
    images=imgs,
    coordinate_systems=(world_cs,),
    coordinate_transformations=transforms
)

scene.to_ome_zarr(os.path.join(r"/vol/data/SCAPE/20260826", "scene.ome.zarr"), compute=True, overwrite=True)

[]

## Detections

In [3]:
from ultralytics import YOLO

In [4]:
model = YOLO(r"train-6_YOLO_ASI/weights/best.pt")

In [ ]:
resolution_level = 2

image_layers = []
shapes_layers = []
features = pd.DataFrame()

scene = OMEZarrScene.from_ome_zarr(os.path.join(r"/vol/data/SCAPE/20260826", "scene.ome.zarr"))

images = scene.images
for name, ms in tqdm.tqdm(images.items()):
    # retrieve translation and multiscales
    translation = [t for t in scene.metadata.coordinateTransformations if t.input.path == ms.name][0]
    translate = translation.translation
    scale=list(ms.images[resolution_level].scale.values())

    projection = ms.images[resolution_level].data.min(axis=0)
    projection = (projection - projection.min()) / (projection.max() - projection.min()) * 255
    z0 = translate[0] + ms.images[resolution_level].data.shape[0] * scale[0] / 2

    df_rectangles = sliding_window_inference(projection.compute(), model, window_size=(640, 640), overlap=0.2)
    df_rectangles = remove_duplicate_detections(df_rectangles, iou_threshold=0.1)
    df_rectangles = extract_features(projection.compute(), df_rectangles)
    
    rectangles = []
    for _, row in df_rectangles.iterrows():
        x0 = float(row['x0']) * scale[2] + translation.translation[2]
        y0 = float(row['y0']) * scale[1] + translation.translation[1]
        x1 = float(row['x1']) * scale[2] + translation.translation[2] 
        y1 = float(row['y1']) * scale[1] + translation.translation[1]
        rectangles.append([(z0, y0, x0), (z0, y0, x1), (z0, y1, x1), (z0, y1, x0)])

    df_rectangles.drop(columns=["x0", "y0", "x1", "y1"], inplace=True)
    df_rectangles["Well"] = ms.name
    # df_rectangles["Row"] = ms.name[0]
    # df_rectangles["Column"] = int(ms.name[1:])

    features = pd.concat([features, df_rectangles], ignore_index=True)

    image_layers.append(
        napari.layers.Image(
            [img.data for img in ms.images],
            name=os.path.basename(ms.name),
            translate=translate,
            scale=list(ms.images[0].scale.values())
        )
    )
    shapes_layers.append(napari.layers.Shapes(
            rectangles, shape_type="rectangle", edge_color="red",
            face_color="transparent", features=df_rectangles, edge_width = 0.00001,
            name=os.path.basename(ms.name) + "_detections"
        )
    )

# [viewer.add_layer(layer) for layer in image_layers]

# concatenate all shapes layers into one
shapes = [shape for layer in shapes_layers for shape in layer.data]

# make these columns categorical
features["Well"] = features["Well"].astype("category")
# features["Row"] = features["Row"].astype("category")
# features["Column"] = features["Column"].astype("category")

# viewer.add_shapes(
# 	shapes, shape_type="rectangle", edge_color="red",
# 	face_color="transparent", features=features, edge_width = 0.00001,
# 	name="all_detections"
# )


## View

In [8]:
viewer = napari.Viewer()
viewer.open(r"/vol/data/SCAPE/20260826/scene.ome.zarr", plugin="napari-ome-zarr")

Root group {'ome': {'scene': {'coordinateTransformations': [{'type': 'translation', 'input': {'name': 'physical', 'path': '20260811_BF_W2_full_ImageStackFile_1'}, 'output': {'name': 'world'}, 'translation': [-53.199999999999996, 9894.38388, 2902.18062]}], 'coordinateSystems': [{'name': 'world', 'axes': [{'name': 'z', 'type': 'space', 'unit': 'micrometer'}, {'name': 'y', 'type': 'space', 'unit': 'micrometer'}, {'name': 'x', 'type': 'space', 'unit': 'micrometer'}]}]}, 'version': '0.6'}}


Opening Files: 1it [00:00,  3.67it/s]


[<Image layer '20260811_BF_W2_full_ImageStackFile_1: Brightfield' at 0x783a60360590>]

In [12]:
# Initialize the S3 client with a custom endpoint
s3 = boto3.client(
    "s3",
    endpoint_url="https://radosgw.public.os.wwu.de",  # e.g., "https://s3.us-west-001.backblazeb2.com"
    aws_access_key_id="D7PVI74O795B3GTG7UYM",
    aws_secret_access_key="xuauB5uR3SGM1SljYOqSLVefDDkhOQD0jUrPgPgF",
)

bucket_name = "methods-paper26"  # Replace with your bucket name
local_zarr = r"/vol/data/SCAPE/20260826/scene.ome.zarr"
# Walk through the local .zarr directory and upload files
iterator = tqdm.tqdm(os.walk(local_zarr), desc="Uploading files to S3")
for root, dirs, files in iterator:
    for file in files:
        local_file_path = os.path.join(root, file)
        # Create the S3 key (path in the bucket)
        relative_path = os.path.relpath(local_file_path, local_zarr)
        s3_key = os.path.join("ASI_plate1_overview_image_20260826.ome.zarr", relative_path).replace("\\", "/")

        s3.upload_file(local_file_path, bucket_name, s3_key)

Uploading files to S3: 56it [07:56,  8.52s/it]


In [21]:
# Initialize the S3 client with a custom endpoint
s3 = boto3.client(
    "s3",
    endpoint_url="https://nbg1.your-objectstorage.com",  # e.g., "https://s3.us-west-001.backblazeb2.com"
    aws_access_key_id="0CQPS18PR6NBT7OYRN0P",
    aws_secret_access_key="m9PJdJej0JypCi8GgDWiVEVAtFRflKBtQIM6p7cA",
)

s3.upload_file(Filename="test.txt", Bucket="gerbi", Key="multimodal-techdev/test.txt")